In [5]:
import os
from dotenv import find_dotenv, load_dotenv

# Always reload latest values from .env into the current notebook kernel.
ENV_PATH = find_dotenv(filename=".env", usecwd=True)
if not ENV_PATH:
    raise FileNotFoundError("Could not locate a .env file from the current working directory tree.")

load_dotenv(dotenv_path=ENV_PATH, override=True)

True

In [ ]:
# OpenAI client setup

from pathlib import Path
from openai import OpenAI
import httpx

cert_env = os.environ.get("CERT_LLM", "").strip()
if not cert_env:
    raise EnvironmentError(
        "CERT_LLM is not set. Point it to your CA bundle file or directory."
    )

cert_path = Path(cert_env).expanduser()
# If CERT_LLM is a directory, use the default bundle filename inside it.
if cert_path.is_dir():
    cert_path = cert_path / "ca-bundle_lite.crt"

SSL_CERT = str(cert_path)
if not cert_path.exists():
    raise FileNotFoundError(f"Certificate bundle not found at: {SSL_CERT}")

BASE_URL = "https://litellm.icp.infineon.com"
MODEL = os.environ.get("OPENAI_MODEL", "gpt-5.2").strip()

http_client = httpx.Client(verify=SSL_CERT)
client = OpenAI(
    base_url=f"{BASE_URL}/v1",
    api_key=os.environ.get("OPENAI_API_KEY"),
    http_client=http_client,
)


In [7]:
model_list = client.models.list()
available_models = [m.id for m in model_list.data if getattr(m, "id", None)]
valid_models = [m for m in available_models if m != "no-default-models"]

if MODEL == "no-default-models":
    MODEL = ""

if MODEL and MODEL not in available_models:
    print(f"Requested model '{MODEL}' is not available for this key.")
    MODEL = valid_models[0] if valid_models else ""

if not MODEL:
    print("No valid model available from the proxy.")
    print("Set OPENAI_MODEL to a model your key can access and rerun this cell.")
    print(f"Models returned by /v1/models: {available_models}")
else:
    print(f"Using model: {MODEL}")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": "this is a test request, write a short poem",
            }
        ],
    )
    

from langchain.chat_models import init_chat_model

# Route through the LiteLLM proxy (reuses the SSL bundle + key); a bare model name
# would default to the public OpenAI endpoint, which the network refuses.
model = init_chat_model(
    MODEL,
    model_provider="openai",
    base_url=f"{BASE_URL}/v1",
    api_key=os.environ.get("OPENAI_API_KEY"),
    http_client=http_client,
)

Using model: gpt-5.2


# Tools

Model can request to call tools that perform tasks such as fetching data from DB, searching the Web, Running code.
Tools are pairing of:
1. A schema  including the name of the tool, Description and/or Arg definition (often a JSON schema)
2. A function or co-routine to execute

In [8]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Hello, how are you?")
response.content


'<think>\nOkay, the user asked "Hello, how are you?" first. I need to respond in a friendly and approachable way. Since I\'m an AI, I can\'t feel emotions, but I can make it clear that I\'m here and ready to help. I should keep the tone positive and open so they feel comfortable asking questions. I\'ll check for any cultural nuances, but since the user is using English and initiating with a standard greeting, I can respond similarly with a greeting and offer assistance. I want to make sure my response is concise and doesn\'t overcomplicate things. Just a simple acknowledgment and an invitation to ask for help. Alright, that should cover it.\n</think>\n\nHello! I\'m doing well, thank you for asking. How can I assist you today? 😊'

### Create a Tool

In [9]:
from langchain.tools import tool


@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    
    # In a real implementation, you would call a weather API here.
    return f"The current weather in {location} is sunny with a temperature of 25°C."


# Bind tool to LLM model

model_tool = model.bind_tools([get_weather])

In [ ]:
response = model_tool.invoke("What's the weather like in New York?")
print(response.content)  # Model's text response (may be empty when it requests a tool).

# `response.tool_calls` is a list of dicts with keys: name, args, id, type.
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']} with input: {tool_call['args']}")
    print(f"Tool call type: {tool_call['type']}")

    # Actually run the requested tool to get its output.
    tool_output = get_weather.invoke(tool_call)
    print(f"Tool output: {tool_output.content}")



Tool called: get_weather with input: {'location': 'New York'}
Tool call id: a6dka90vg
Tool call type: tool_call
Tool output: The current weather in New York is sunny with a temperature of 25°C.


### Tool execution loop

In [13]:
# Step 1: model generate tool calls
messages = [
    {"role": "user", "content": "What's the weather like in New York?"}
]

ai_msg = model_tool.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tool and collect results
for tool_call in ai_msg.tool_calls:
    print(f"Tool called: {tool_call['name']} with input: {tool_call['args']}")
    print(f"Tool call type: {tool_call['type']}")

    # Actually run the requested tool to get its output.
    tool_output = get_weather.invoke(tool_call)
    print(f"Tool output: {tool_output.content}")

    # Append the tool's output to the messages for the model to see.
    messages.append(tool_output)
    
# Step 3: Pass results back to model for final response
final_response = model_tool.invoke(messages)
print(final_response.content)  # Model's final text response after seeing the tool output.
    

Tool called: get_weather with input: {'location': 'New York'}
Tool call type: tool_call
Tool output: The current weather in New York is sunny with a temperature of 25°C.
The current weather in New York is sunny with a temperature of 25°C. It's a pleasant day! 🌞


In [15]:
# Step by step execution of tool calls with explicit
messages

[{'role': 'user', 'content': "What's the weather like in New York?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in New York. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is provided as "New York". So I\'ll call the function with that location. Make sure the JSON is correctly formatted with the name and arguments.\n', 'tool_calls': [{'id': '50x34ewzy', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 157, 'total_tokens': 247, 'completion_time': 0.145231312, 'completion_tokens_details': {'reasoning_tokens': 65}, 'prompt_time': 0.006184308, 'prompt_tokens_details': None, 'queue_time': 0.052403001, 'total_time': 0.15141562}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason'